In [1]:
!pip install -q qiskit qiskit-aer matplotlib numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.8/9.8 MB 42.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 43.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 30.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.9/54.9 kB 3.3 MB/s eta 0:00:00


In [2]:

from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator

n = 3
circuit = QuantumCircuit(4, 3)

# Set the last qubit to |1>
circuit.x(3)

# Create superposition
for i in range(4):
    circuit.h(i)

# Oracle for f(x) = 0
# Identity operation; no gate is applied

# Transform the first three qubits
for i in [0, 1, 2]:
    circuit.h(i)

# Record measurements
circuit.measure([0, 1, 2], [0, 1, 2])

print(circuit.draw())

sim = AerSimulator()
job = sim.run(circuit, shots=1024)
result = job.result()

counts = result.get_counts()
print("Output counts:", counts)


     ┌───┐┌───┐┌─┐      
q_0: ┤ H ├┤ H ├┤M├──────
     ├───┤├───┤└╥┘┌─┐   
q_1: ┤ H ├┤ H ├─╫─┤M├───
     ├───┤├───┤ ║ └╥┘┌─┐
q_2: ┤ H ├┤ H ├─╫──╫─┤M├
     ├───┤├───┤ ║  ║ └╥┘
q_3: ┤ X ├┤ H ├─╫──╫──╫─
     └───┘└───┘ ║  ║  ║ 
c: 3/═══════════╩══╩══╩═
                0  1  2 
Output counts: {'000': 1024}


In [3]:

from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator

n = 3
qc = QuantumCircuit(4, 3)

# Prepare the target qubit
qc.x(3)

# Create superposition across all qubits
for q in range(4):
    qc.h(q)

# Implement the balanced parity oracle
qc.cx(0, 3)
qc.cx(1, 3)
qc.cx(2, 3)

# Apply the final Hadamard transformation
qc.h(0)
qc.h(1)
qc.h(2)

# Measure the input qubits
qc.measure([0, 1, 2], [0, 1, 2])

print(qc.draw())

sim = AerSimulator()
job = sim.run(qc, shots=1024)
result = job.result()

counts = result.get_counts()
print("Measurement results:", counts)


     ┌───┐          ┌───┐     ┌─┐           
q_0: ┤ H ├───────■──┤ H ├─────┤M├───────────
     ├───┤       │  └───┘┌───┐└╥┘     ┌─┐   
q_1: ┤ H ├───────┼────■──┤ H ├─╫──────┤M├───
     ├───┤       │    │  └───┘ ║ ┌───┐└╥┘┌─┐
q_2: ┤ H ├───────┼────┼────■───╫─┤ H ├─╫─┤M├
     ├───┤┌───┐┌─┴─┐┌─┴─┐┌─┴─┐ ║ └───┘ ║ └╥┘
q_3: ┤ X ├┤ H ├┤ X ├┤ X ├┤ X ├─╫───────╫──╫─
     └───┘└───┘└───┘└───┘└───┘ ║       ║  ║ 
c: 3/══════════════════════════╩═══════╩══╩═
                               0       1  2 
Measurement results: {'111': 1024}


In [4]:

from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator

def create_dj_circuit(n, mode):
    qc = QuantumCircuit(n + 1, n)

    # Initialize the target qubit
    qc.x(n)

    # Create superposition
    for q in range(n + 1):
        qc.h(q)

    # Implement the selected oracle
    if mode == "constant":
        # No oracle gate required
        pass
    elif mode == "balanced":
        for q in range(n):
            qc.cx(q, n)
    else:
        raise ValueError("Mode must be constant or balanced")

    # Transform the input register
    for q in range(n):
        qc.h(q)

    qc.measure(list(range(n)), list(range(n)))
    return qc


def simulate_circuit(circuit, backend):
    job = backend.run(circuit, shots=1024)
    return job.result().get_counts()


backend = AerSimulator()

for qubit_count in [2, 3, 4, 5]:
    print("\nInput qubits:", qubit_count)

    for mode in ["constant", "balanced"]:
        circuit = create_dj_circuit(qubit_count, mode)
        output = simulate_circuit(circuit, backend)

        print(mode, "->", output)



Input qubits: 2
constant -> {'00': 1024}
balanced -> {'11': 1024}

Input qubits: 3
constant -> {'000': 1024}
balanced -> {'111': 1024}

Input qubits: 4
constant -> {'0000': 1024}
balanced -> {'1111': 1024}

Input qubits: 5
constant -> {'00000': 1024}
balanced -> {'11111': 1024}
